# Fase 2.3 — Modelo TTM (Tiny Time Mixers — IBM Granite)

**Thesis:** Comparativa de algoritmos para la predicción de la demanda eléctrica  
**Author:** Antonio Navarro  

This notebook evaluates the IBM Granite Tiny Time Mixers foundation model
(`ibm/granite-timeseries-ttm-r2`) in two modes:

1. **Zero-shot** — load the pre-trained model, run inference with no fine-tuning
2. **Few-shot** — fine-tune on 5% of training data, then evaluate on the test set

The model context length is **512 hours** → forecast horizon **24 hours**.

### Installation (run once in your venv)
```bash
pip install "granite-tsfm[notebooks]>=0.2" transformers>=4.40 datasets>=2.14 accelerate>=0.27
```

In [1]:
# Install dependencies not pre-installed in Colab
import importlib, subprocess, sys

def _pip(pkg, import_name=None):
    if importlib.util.find_spec(import_name or pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_pip("granite-tsfm[notebooks]", "tsfm_public")
_pip("transformers")
_pip("accelerate")
print("Dependencies ready.")

# Fase 2.3 — Modelo TTM (Tiny Time Mixers — IBM Granite)

**Thesis:** Comparativa de algoritmos para la predicción de la demanda eléctrica  
**Author:** Antonio Navarro  

This notebook evaluates the IBM Granite Tiny Time Mixers foundation model
(`ibm/granite-timeseries-ttm-r2`) in two modes:

1. **Zero-shot** — load the pre-trained model, run inference with no fine-tuning
2. **Few-shot** — fine-tune on 5% of training data, then evaluate on the test set

The model context length is **512 hours** → forecast horizon **24 hours**.

### Installation (run once in your venv)
```bash
pip install "granite-tsfm[notebooks]>=0.2" transformers>=4.40 datasets>=2.14 accelerate>=0.27
```

# Fase 2.3 — Modelo TTM (Tiny Time Mixers — IBM Granite)

**Thesis:** Comparativa de algoritmos para la predicción de la demanda eléctrica  
**Author:** Antonio Navarro  

This notebook evaluates the IBM Granite Tiny Time Mixers foundation model
(`ibm/granite-timeseries-ttm-r2`) in two modes:

1. **Zero-shot** — load the pre-trained model, run inference with no fine-tuning
2. **Few-shot** — fine-tune on 5% of training data, then evaluate on the test set

The model context length is **512 hours** → forecast horizon **24 hours**.

### Installation (run once in your venv)
```bash
pip install "granite-tsfm[notebooks]>=0.2" transformers>=4.40 datasets>=2.14 accelerate>=0.27
```

In [ ]:
# Install dependencies not pre-installed in Colab
import importlib, subprocess, sys

def _pip(pkg, import_name=None):
    if importlib.util.find_spec(import_name or pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_pip("granite-tsfm[notebooks]", "tsfm_public")
_pip("transformers")
_pip("accelerate")
print("Dependencies ready.")

In [ ]:
import sys, os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import torch

# ── Colab: clone or update the repo, then cd into it ──────────────────────────
REPO_NAME = "pred_demanda"
REPO_URL  = "https://github.com/antonionc/pred_demanda.git"

if os.path.exists("data_utils.py"):
    !git pull
elif os.path.isdir(REPO_NAME):
    %cd {REPO_NAME}
    !git pull
elif os.path.exists(f"/content/{REPO_NAME}"):
    %cd /content/{REPO_NAME}
    !git pull
elif os.path.exists("/content"):
    !git clone {REPO_URL}
    %cd {REPO_NAME}
# ──────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, os.path.abspath('.'))
import data_utils as du

# Mount Google Drive in Colab and create persistent links for data/, cache/, saved_models/
du.setup_colab_drive()

# Set up Hugging Face token (reads .env / environment / Colab UI safely)
du.setup_hf_token()

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# granite-tsfm imports
try:
    from tsfm_public.models.tinytimemixer import (
        TinyTimeMixerForPrediction,
        TinyTimeMixerConfig,
    )
    from tsfm_public import TinyTimeMixerForPrediction  # noqa (alias)
    from tsfm_public.toolkit.dataset import ForecastDFDataset
    from tsfm_public.toolkit.time_series_forecasting_pipeline import TimeSeriesForecastingPipeline
    from transformers import EarlyStoppingCallback, Trainer, TrainingArguments
    print('granite-tsfm imported successfully')
except ImportError as e:
    print(f'Import error: {e}')
    print('Install with: pip install "granite-tsfm[notebooks]"')
    raise

## 1 · Load and prepare data

In [ ]:
df_train = pd.read_csv('data/features_train.csv', parse_dates=['datetime'])
df_val   = pd.read_csv('data/features_val.csv',   parse_dates=['datetime'])
df_test  = pd.read_csv('data/features_test.csv',  parse_dates=['datetime'])

# TTM model parameters
CONTEXT_LEN = 512    # 512-hour historical context window (~21 days)
HORIZON     = 24     # 24-hour forecast horizon
TARGET_COL  = 'demand_mw'

# TTMs work best with univariate signals, so we remove the noise (exogenous variables)

print(f'Target variable : {TARGET_COL}')
print(f'Train : {len(df_train):,}  Val : {len(df_val):,}  Test : {len(df_test):,}')

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardize target channel
scaler = StandardScaler()
scaler.fit(df_train[[TARGET_COL]])

def scale_df(df):
    out = df.copy()
    out[[TARGET_COL]] = scaler.transform(df[[TARGET_COL]])
    return out

df_train_sc = scale_df(df_train)
df_val_sc   = scale_df(df_val)
df_test_sc  = scale_df(df_test)

# Concatenated history for zero-shot inference (train + val as context)
df_history = pd.concat([df_train_sc, df_val_sc], ignore_index=True)
print(f'History length for zero-shot: {len(df_history):,} rows')

In [ ]:
TTM_MODEL_ID = 'ibm-granite/granite-timeseries-ttm-r2'

print(f'Loading {TTM_MODEL_ID}…')
zs_model = TinyTimeMixerForPrediction.from_pretrained(
    TTM_MODEL_ID,
    prediction_filter_length = HORIZON,
)
zs_model = zs_model.to(DEVICE)
zs_model.eval()

config = zs_model.config
print(f'Context length : {config.context_length}')
print(f'Forecast length: {config.prediction_length}')
print(f'Parameters     : {sum(p.numel() for p in zs_model.parameters()):,}')

## 3 · Build ForecastDFDatasets

In [ ]:
def make_dataset(df, context_len=CONTEXT_LEN, horizon=HORIZON, stride=1):
    """Wrap a DataFrame in ForecastDFDataset expected by granite-tsfm."""
    return ForecastDFDataset(
        df,
        id_columns           = [],
        timestamp_column     = 'datetime',
        target_columns       = [TARGET_COL],
        conditional_columns  = [],
        context_length       = context_len,
        prediction_length    = horizon,
        stride               = stride,
    )

# Zero-shot test dataset: use last CONTEXT_LEN rows of history as prefix,
# then test data — ensures every prediction window falls within the test period.
df_for_zs_test = pd.concat([
    df_history.tail(CONTEXT_LEN).reset_index(drop=True),
    df_test_sc,
], ignore_index=True)

# stride=HORIZON gives non-overlapping daily predictions
test_dataset_zs = make_dataset(df_for_zs_test, stride=HORIZON)
print(f'Zero-shot test dataset: {len(test_dataset_zs)} windows')

# Few-shot fine-tuning dataset: sample 5% of windows uniformly across all 3.5 years (stride=24)
fs_train_ds = make_dataset(df_train_sc, stride=24)
fs_val_ds   = make_dataset(df_val_sc,   stride=HORIZON)
print(f'Few-shot train dataset  : {len(fs_train_ds)} windows (sampled across full 3.5-year history)')
print(f'Few-shot val dataset    : {len(fs_val_ds)} windows')

In [ ]:
from torch.utils.data import DataLoader
from torch.utils.data.dataloader import default_collate

def collate_skip_timestamps(batch):
    """Drop pandas Timestamp fields that PyTorch's default collate cannot handle."""
    filtered = [
        {k: v for k, v in item.items() if not isinstance(v, pd.Timestamp)}
        for item in batch
    ]
    return default_collate(filtered)

t0  = time.time()
loader = DataLoader(test_dataset_zs, batch_size=64, shuffle=False,
                    collate_fn=collate_skip_timestamps)

preds_zs, trues_zs = [], []

with torch.no_grad():
    for batch in loader:
        past_values = batch['past_values'].to(DEVICE)
        outputs     = zs_model(past_values=past_values)
        preds_zs.append(outputs.prediction_outputs.cpu().numpy())
        trues_zs.append(batch['future_values'].numpy())

inference_time_zs = time.time() - t0

preds_zs = np.concatenate(preds_zs, axis=0)   # (n_windows, horizon, 1)
trues_zs = np.concatenate(trues_zs, axis=0)

# Extract target channel (index 0) and inverse-transform
target_mean = scaler.mean_[0]
target_std  = scaler.scale_[0]

y_pred_zs = preds_zs[:, :, 0].ravel() * target_std + target_mean
y_true_zs = trues_zs[:, :, 0].ravel() * target_std + target_mean

print(f'Inference time (zero-shot): {inference_time_zs:.2f}s')
print(f'Forecast points: {len(y_pred_zs):,}')

In [ ]:
metrics_zs = du.compute_metrics(y_true_zs, y_pred_zs, label='TTM zero-shot')
metrics_zs['train_s']     = 0.0
metrics_zs['inference_s'] = inference_time_zs

## 5 · Few-shot fine-tuning (5% of training data)

In [ ]:
# REMOVE/comment later
# debugging an issue with the TRainingArguments consutructor
import inspect
import transformers

print("Transformers version:", transformers.__version__)
print("TrainingArguments class :", TrainingArguments)
print("Defined in module       :", TrainingArguments.__module__)
try:
    print("Source file             :", inspect.getfile(TrainingArguments))
except Exception:
    pass
print("\nAccepted constructor parameters:")
print(inspect.signature(TrainingArguments.__init__))

In [ ]:
# Load pre-trained model in native Channel-Independent mode (preserves zero-shot curve)
fs_model = TinyTimeMixerForPrediction.from_pretrained(
    TTM_MODEL_ID,
    prediction_filter_length = HORIZON,
)

# Gentle fine-tuning: allow small learning rate adaptation across the model
# (or train prediction head with pre-trained weights intact)
for param in fs_model.backbone.parameters():
    param.requires_grad = True

print(f'Trainable parameters: {sum(p.numel() for p in fs_model.parameters() if p.requires_grad):,}')

training_args = TrainingArguments(
    output_dir                  = 'saved_models/ttm_fewshot',
    num_train_epochs            = 10,
    per_device_train_batch_size = 64,
    per_device_eval_batch_size  = 64,
    learning_rate               = 2e-5,                 # Gentle learning rate for foundation adaptation
    lr_scheduler_type           = 'cosine',
    warmup_steps                = 10,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    save_total_limit            = 2,
    logging_steps               = 5,
    report_to                   = 'none',
    seed                        = 42,
)

trainer = Trainer(
    model        = fs_model,
    args         = training_args,
    train_dataset= fs_train_ds,
    eval_dataset = fs_val_ds,
    callbacks    = [
        EarlyStoppingCallback(
            early_stopping_patience  = 3,
            early_stopping_threshold = 0.0005,
        )
    ],
)

print(f'Fine-tuning on {len(fs_train_ds)} windows…')
t0 = time.time()
trainer.train()
finetune_time = time.time() - t0
print(f'Fine-tuning completed in {finetune_time:.1f}s  ({finetune_time/60:.1f} min)')

In [ ]:
# Plot fine-tuning loss
log_history = trainer.state.log_history
train_losses = [e['loss'] for e in log_history if 'loss' in e and 'eval_loss' not in e]
eval_losses  = [e['eval_loss'] for e in log_history if 'eval_loss' in e]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label='Train loss')
if eval_losses:
    eval_x = np.linspace(0, len(train_losses)-1, len(eval_losses))
    ax.plot(eval_x, eval_losses, label='Val loss', linestyle='--')
ax.set_title('TTM few-shot fine-tuning loss')
ax.set_xlabel('Step')
ax.legend()
plt.tight_layout()
plt.savefig('data/fig_ttm_finetune.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fs_model.eval()
fs_model = fs_model.to(DEVICE)

# Re-use the same test dataset built for zero-shot
fs_loader = DataLoader(test_dataset_zs, batch_size=64, shuffle=False,
                       collate_fn=collate_skip_timestamps)


t0 = time.time()
preds_fs, trues_fs = [], []

with torch.no_grad():
    for batch in fs_loader:
        past_values = batch['past_values'].to(DEVICE)
        outputs     = fs_model(past_values=past_values)
        preds_fs.append(outputs.prediction_outputs.cpu().numpy())
        trues_fs.append(batch['future_values'].numpy())

inference_time_fs = time.time() - t0

preds_fs = np.concatenate(preds_fs, axis=0)
trues_fs = np.concatenate(trues_fs, axis=0)

y_pred_fs = preds_fs[:, :, 0].ravel() * target_std + target_mean
y_true_fs = trues_fs[:, :, 0].ravel() * target_std + target_mean

print(f'Inference time (few-shot): {inference_time_fs:.2f}s')
print(f'Forecast points: {len(y_pred_fs):,}')

In [ ]:
metrics_fs = du.compute_metrics(y_true_fs, y_pred_fs, label='TTM few-shot')
metrics_fs['train_s']     = finetune_time
metrics_fs['inference_s'] = inference_time_fs

## 7 · Plots

In [ ]:
# Build datetime array for test predictions
n_windows  = len(preds_zs)
dates_test = df_for_zs_test['datetime'].values

dates_zs = np.concatenate([
    dates_test[CONTEXT_LEN + i * HORIZON : CONTEXT_LEN + i * HORIZON + HORIZON]
    for i in range(n_windows)
])
dates_zs = pd.to_datetime(dates_zs)

In [ ]:
du.plot_predictions(
    y_true_zs[:168], y_pred_zs[:168],
    title     = 'TTM zero-shot — first week of test set',
    dates     = dates_zs[:168],
    save_path = 'data/fig_ttm_zeroshot_week.png',
)

In [ ]:
du.plot_predictions(
    y_true_fs[:168], y_pred_fs[:168],
    title     = 'TTM few-shot — first week of test set',
    dates     = dates_zs[:168],
    save_path = 'data/fig_ttm_fewshot_week.png',
)

In [ ]:
# Side-by-side zero-shot vs few-shot (one representative week)
idx = slice(0, 168)
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, y_pred, label, color in [
    (axes[0], y_pred_zs[idx], 'Zero-shot', '#ff7f0e'),
    (axes[1], y_pred_fs[idx], 'Few-shot',  '#2ca02c'),
]:
    ax.plot(dates_zs[idx], y_true_zs[idx], label='Actual',  color='#1f77b4', linewidth=1.2)
    ax.plot(dates_zs[idx], y_pred,          label=label,   color=color, linewidth=1.2, linestyle='--')
    ax.set_title(f'TTM {label}')
    ax.set_ylabel('Demand (MW)')
    ax.legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.tight_layout()
plt.savefig('data/fig_ttm_comparison_week.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 · Save predictions and metrics

In [ ]:
# Zero-shot
pd.DataFrame({'datetime': dates_zs, 'y_true': y_true_zs, 'y_pred': y_pred_zs}).to_csv(
    'data/predictions_ttm_zeroshot.csv', index=False
)
with open('data/metrics_ttm_zeroshot.json', 'w') as f:
    json.dump(metrics_zs, f, indent=2)

# Few-shot
pd.DataFrame({'datetime': dates_zs, 'y_true': y_true_fs, 'y_pred': y_pred_fs}).to_csv(
    'data/predictions_ttm_fewshot.csv', index=False
)
with open('data/metrics_ttm_fewshot.json', 'w') as f:
    json.dump(metrics_fs, f, indent=2)

print('Saved predictions and metrics for both TTM modes.')
print('\nZero-shot metrics:', metrics_zs)
print('Few-shot  metrics:', metrics_fs)

# Sync outputs to Google Drive (if in Colab)
du.sync_to_drive()


## Summary

| Mode | MAPE (%) | RMSE (MW) | MAE (MW) | Fine-tune time |
|------|----------|-----------|----------|----------------|
| Zero-shot | … | … | … | 0 s |
| Few-shot  | … | … | … | … s |

Next step → `fase3_validacion_y_comparativa.ipynb`

In [ ]:
#Terminate runtime
from google.colab import runtime
runtime.unassign()

In [ ]:
# Install dependencies not pre-installed in Colab
import importlib, subprocess, sys

def _pip(pkg, import_name=None):
    if importlib.util.find_spec(import_name or pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_pip("granite-tsfm[notebooks]", "tsfm_public")
_pip("transformers")
_pip("accelerate")
print("Dependencies ready.")

In [ ]:
import sys, os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import torch

# ── Colab: clone or update the repo, then cd into it ──────────────────────────
REPO_NAME = "pred_demanda"
REPO_URL  = "https://github.com/antonionc/pred_demanda.git"

if os.path.exists("data_utils.py"):
    !git pull
elif os.path.isdir(REPO_NAME):
    %cd {REPO_NAME}
    !git pull
elif os.path.exists(f"/content/{REPO_NAME}"):
    %cd /content/{REPO_NAME}
    !git pull
elif os.path.exists("/content"):
    !git clone {REPO_URL}
    %cd {REPO_NAME}
# ──────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, os.path.abspath('.'))
import data_utils as du

# Mount Google Drive in Colab and create persistent links for data/, cache/, saved_models/
du.setup_colab_drive()

# Set up Hugging Face token (reads .env / environment / Colab UI safely)
du.setup_hf_token()

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# granite-tsfm imports
try:
    from tsfm_public.models.tinytimemixer import (
        TinyTimeMixerForPrediction,
        TinyTimeMixerConfig,
    )
    from tsfm_public import TinyTimeMixerForPrediction  # noqa (alias)
    from tsfm_public.toolkit.dataset import ForecastDFDataset
    from tsfm_public.toolkit.time_series_forecasting_pipeline import TimeSeriesForecastingPipeline
    from transformers import EarlyStoppingCallback, Trainer, TrainingArguments
    print('granite-tsfm imported successfully')
except ImportError as e:
    print(f'Import error: {e}')
    print('Install with: pip install "granite-tsfm[notebooks]"')
    raise

## 1 · Load and prepare data

In [ ]:
df_train = pd.read_csv('data/features_train.csv', parse_dates=['datetime'])
df_val   = pd.read_csv('data/features_val.csv',   parse_dates=['datetime'])
df_test  = pd.read_csv('data/features_test.csv',  parse_dates=['datetime'])

# TTM model parameters
CONTEXT_LEN = 512    # 512-hour historical context window (~21 days)
HORIZON     = 24     # 24-hour forecast horizon
TARGET_COL  = 'demand_mw'

# TTMs work best with univariate signals, so we remove the noise (exogenous variables)

print(f'Target variable : {TARGET_COL}')
print(f'Train : {len(df_train):,}  Val : {len(df_val):,}  Test : {len(df_test):,}')

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardize target channel
scaler = StandardScaler()
scaler.fit(df_train[[TARGET_COL]])

def scale_df(df):
    out = df.copy()
    out[[TARGET_COL]] = scaler.transform(df[[TARGET_COL]])
    return out

df_train_sc = scale_df(df_train)
df_val_sc   = scale_df(df_val)
df_test_sc  = scale_df(df_test)

# Concatenated history for zero-shot inference (train + val as context)
df_history = pd.concat([df_train_sc, df_val_sc], ignore_index=True)
print(f'History length for zero-shot: {len(df_history):,} rows')

In [ ]:
TTM_MODEL_ID = 'ibm-granite/granite-timeseries-ttm-r2'

print(f'Loading {TTM_MODEL_ID}…')
zs_model = TinyTimeMixerForPrediction.from_pretrained(
    TTM_MODEL_ID,
    prediction_filter_length = HORIZON,
)
zs_model = zs_model.to(DEVICE)
zs_model.eval()

config = zs_model.config
print(f'Context length : {config.context_length}')
print(f'Forecast length: {config.prediction_length}')
print(f'Parameters     : {sum(p.numel() for p in zs_model.parameters()):,}')

## 3 · Build ForecastDFDatasets

In [ ]:
def make_dataset(df, context_len=CONTEXT_LEN, horizon=HORIZON, stride=1):
    """Wrap a DataFrame in ForecastDFDataset expected by granite-tsfm."""
    return ForecastDFDataset(
        df,
        id_columns           = [],
        timestamp_column     = 'datetime',
        target_columns       = [TARGET_COL],
        conditional_columns  = [],
        context_length       = context_len,
        prediction_length    = horizon,
        stride               = stride,
    )

# Zero-shot test dataset: use last CONTEXT_LEN rows of history as prefix,
# then test data — ensures every prediction window falls within the test period.
df_for_zs_test = pd.concat([
    df_history.tail(CONTEXT_LEN).reset_index(drop=True),
    df_test_sc,
], ignore_index=True)

# stride=HORIZON gives non-overlapping daily predictions
test_dataset_zs = make_dataset(df_for_zs_test, stride=HORIZON)
print(f'Zero-shot test dataset: {len(test_dataset_zs)} windows')

# Few-shot fine-tuning dataset: sample 5% of windows uniformly across all 3.5 years (stride=24)
fs_train_ds = make_dataset(df_train_sc, stride=24)
fs_val_ds   = make_dataset(df_val_sc,   stride=HORIZON)
print(f'Few-shot train dataset  : {len(fs_train_ds)} windows (sampled across full 3.5-year history)')
print(f'Few-shot val dataset    : {len(fs_val_ds)} windows')

In [ ]:
from torch.utils.data import DataLoader
from torch.utils.data.dataloader import default_collate

def collate_skip_timestamps(batch):
    """Drop pandas Timestamp fields that PyTorch's default collate cannot handle."""
    filtered = [
        {k: v for k, v in item.items() if not isinstance(v, pd.Timestamp)}
        for item in batch
    ]
    return default_collate(filtered)

t0  = time.time()
loader = DataLoader(test_dataset_zs, batch_size=64, shuffle=False,
                    collate_fn=collate_skip_timestamps)

preds_zs, trues_zs = [], []

with torch.no_grad():
    for batch in loader:
        past_values = batch['past_values'].to(DEVICE)
        outputs     = zs_model(past_values=past_values)
        preds_zs.append(outputs.prediction_outputs.cpu().numpy())
        trues_zs.append(batch['future_values'].numpy())

inference_time_zs = time.time() - t0

preds_zs = np.concatenate(preds_zs, axis=0)   # (n_windows, horizon, 1)
trues_zs = np.concatenate(trues_zs, axis=0)

# Extract target channel (index 0) and inverse-transform
target_mean = scaler.mean_[0]
target_std  = scaler.scale_[0]

y_pred_zs = preds_zs[:, :, 0].ravel() * target_std + target_mean
y_true_zs = trues_zs[:, :, 0].ravel() * target_std + target_mean

print(f'Inference time (zero-shot): {inference_time_zs:.2f}s')
print(f'Forecast points: {len(y_pred_zs):,}')

In [ ]:
metrics_zs = du.compute_metrics(y_true_zs, y_pred_zs, label='TTM zero-shot')
metrics_zs['train_s']     = 0.0
metrics_zs['inference_s'] = inference_time_zs

## 5 · Few-shot fine-tuning (5% of training data)

In [ ]:
# REMOVE/comment later
# debugging an issue with the TRainingArguments consutructor
import inspect
import transformers

print("Transformers version:", transformers.__version__)
print("TrainingArguments class :", TrainingArguments)
print("Defined in module       :", TrainingArguments.__module__)
try:
    print("Source file             :", inspect.getfile(TrainingArguments))
except Exception:
    pass
print("\nAccepted constructor parameters:")
print(inspect.signature(TrainingArguments.__init__))

In [ ]:
# Load pre-trained model in native Channel-Independent mode (preserves zero-shot curve)
fs_model = TinyTimeMixerForPrediction.from_pretrained(
    TTM_MODEL_ID,
    prediction_filter_length = HORIZON,
)

# Gentle fine-tuning: allow small learning rate adaptation across the model
# (or train prediction head with pre-trained weights intact)
for param in fs_model.backbone.parameters():
    param.requires_grad = True

print(f'Trainable parameters: {sum(p.numel() for p in fs_model.parameters() if p.requires_grad):,}')

training_args = TrainingArguments(
    output_dir                  = 'saved_models/ttm_fewshot',
    num_train_epochs            = 10,
    per_device_train_batch_size = 64,
    per_device_eval_batch_size  = 64,
    learning_rate               = 2e-5,                 # Gentle learning rate for foundation adaptation
    lr_scheduler_type           = 'cosine',
    warmup_steps                = 10,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    save_total_limit            = 2,
    logging_steps               = 5,
    report_to                   = 'none',
    seed                        = 42,
)

trainer = Trainer(
    model        = fs_model,
    args         = training_args,
    train_dataset= fs_train_ds,
    eval_dataset = fs_val_ds,
    callbacks    = [
        EarlyStoppingCallback(
            early_stopping_patience  = 3,
            early_stopping_threshold = 0.0005,
        )
    ],
)

print(f'Fine-tuning on {len(fs_train_ds)} windows…')
t0 = time.time()
trainer.train()
finetune_time = time.time() - t0
print(f'Fine-tuning completed in {finetune_time:.1f}s  ({finetune_time/60:.1f} min)')

In [ ]:
# Plot fine-tuning loss
log_history = trainer.state.log_history
train_losses = [e['loss'] for e in log_history if 'loss' in e and 'eval_loss' not in e]
eval_losses  = [e['eval_loss'] for e in log_history if 'eval_loss' in e]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label='Train loss')
if eval_losses:
    eval_x = np.linspace(0, len(train_losses)-1, len(eval_losses))
    ax.plot(eval_x, eval_losses, label='Val loss', linestyle='--')
ax.set_title('TTM few-shot fine-tuning loss')
ax.set_xlabel('Step')
ax.legend()
plt.tight_layout()
plt.savefig('data/fig_ttm_finetune.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fs_model.eval()
fs_model = fs_model.to(DEVICE)

# Re-use the same test dataset built for zero-shot
fs_loader = DataLoader(test_dataset_zs, batch_size=64, shuffle=False,
                       collate_fn=collate_skip_timestamps)


t0 = time.time()
preds_fs, trues_fs = [], []

with torch.no_grad():
    for batch in fs_loader:
        past_values = batch['past_values'].to(DEVICE)
        outputs     = fs_model(past_values=past_values)
        preds_fs.append(outputs.prediction_outputs.cpu().numpy())
        trues_fs.append(batch['future_values'].numpy())

inference_time_fs = time.time() - t0

preds_fs = np.concatenate(preds_fs, axis=0)
trues_fs = np.concatenate(trues_fs, axis=0)

y_pred_fs = preds_fs[:, :, 0].ravel() * target_std + target_mean
y_true_fs = trues_fs[:, :, 0].ravel() * target_std + target_mean

print(f'Inference time (few-shot): {inference_time_fs:.2f}s')
print(f'Forecast points: {len(y_pred_fs):,}')

In [ ]:
metrics_fs = du.compute_metrics(y_true_fs, y_pred_fs, label='TTM few-shot')
metrics_fs['train_s']     = finetune_time
metrics_fs['inference_s'] = inference_time_fs

## 7 · Plots

In [ ]:
# Build datetime array for test predictions
n_windows  = len(preds_zs)
dates_test = df_for_zs_test['datetime'].values

dates_zs = np.concatenate([
    dates_test[CONTEXT_LEN + i * HORIZON : CONTEXT_LEN + i * HORIZON + HORIZON]
    for i in range(n_windows)
])
dates_zs = pd.to_datetime(dates_zs)

In [ ]:
du.plot_predictions(
    y_true_zs[:168], y_pred_zs[:168],
    title     = 'TTM zero-shot — first week of test set',
    dates     = dates_zs[:168],
    save_path = 'data/fig_ttm_zeroshot_week.png',
)

In [ ]:
du.plot_predictions(
    y_true_fs[:168], y_pred_fs[:168],
    title     = 'TTM few-shot — first week of test set',
    dates     = dates_zs[:168],
    save_path = 'data/fig_ttm_fewshot_week.png',
)

In [ ]:
# Side-by-side zero-shot vs few-shot (one representative week)
idx = slice(0, 168)
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, y_pred, label, color in [
    (axes[0], y_pred_zs[idx], 'Zero-shot', '#ff7f0e'),
    (axes[1], y_pred_fs[idx], 'Few-shot',  '#2ca02c'),
]:
    ax.plot(dates_zs[idx], y_true_zs[idx], label='Actual',  color='#1f77b4', linewidth=1.2)
    ax.plot(dates_zs[idx], y_pred,          label=label,   color=color, linewidth=1.2, linestyle='--')
    ax.set_title(f'TTM {label}')
    ax.set_ylabel('Demand (MW)')
    ax.legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.tight_layout()
plt.savefig('data/fig_ttm_comparison_week.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 · Save predictions and metrics

In [ ]:
# Zero-shot
pd.DataFrame({'datetime': dates_zs, 'y_true': y_true_zs, 'y_pred': y_pred_zs}).to_csv(
    'data/predictions_ttm_zeroshot.csv', index=False
)
with open('data/metrics_ttm_zeroshot.json', 'w') as f:
    json.dump(metrics_zs, f, indent=2)

# Few-shot
pd.DataFrame({'datetime': dates_zs, 'y_true': y_true_fs, 'y_pred': y_pred_fs}).to_csv(
    'data/predictions_ttm_fewshot.csv', index=False
)
with open('data/metrics_ttm_fewshot.json', 'w') as f:
    json.dump(metrics_fs, f, indent=2)

print('Saved predictions and metrics for both TTM modes.')
print('\nZero-shot metrics:', metrics_zs)
print('Few-shot  metrics:', metrics_fs)

# Sync outputs to Google Drive (if in Colab)
du.sync_to_drive()


## Summary

| Mode | MAPE (%) | RMSE (MW) | MAE (MW) | Fine-tune time |
|------|----------|-----------|----------|----------------|
| Zero-shot | … | … | … | 0 s |
| Few-shot  | … | … | … | … s |

Next step → `fase3_validacion_y_comparativa.ipynb`

In [ ]:
#Terminate runtime
from google.colab import runtime
runtime.unassign()

In [2]:
import sys, os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import torch

# ── Colab: clone or update the repo, then cd into it ──────────────────────────
REPO_NAME = "pred_demanda"
REPO_URL  = "https://github.com/antonionc/pred_demanda.git"

if os.path.exists("data_utils.py"):
    !git pull
elif os.path.isdir(REPO_NAME):
    %cd {REPO_NAME}
    !git pull
elif os.path.exists(f"/content/{REPO_NAME}"):
    %cd /content/{REPO_NAME}
    !git pull
elif os.path.exists("/content"):
    !git clone {REPO_URL}
    %cd {REPO_NAME}
# ──────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, os.path.abspath('.'))
import data_utils as du

# Mount Google Drive in Colab and create persistent links for data/, cache/, saved_models/
du.setup_colab_drive()

# Set up Hugging Face token (reads .env / environment / Colab UI safely)
du.setup_hf_token()

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [3]:
# granite-tsfm imports
try:
    from tsfm_public.models.tinytimemixer import (
        TinyTimeMixerForPrediction,
        TinyTimeMixerConfig,
    )
    from tsfm_public import TinyTimeMixerForPrediction  # noqa (alias)
    from tsfm_public.toolkit.dataset import ForecastDFDataset
    from tsfm_public.toolkit.time_series_forecasting_pipeline import TimeSeriesForecastingPipeline
    from transformers import EarlyStoppingCallback, Trainer, TrainingArguments
    print('granite-tsfm imported successfully')
except ImportError as e:
    print(f'Import error: {e}')
    print('Install with: pip install "granite-tsfm[notebooks]"')
    raise

## 1 · Load and prepare data

In [4]:
df_train = pd.read_csv('data/features_train.csv', parse_dates=['datetime'])
df_val   = pd.read_csv('data/features_val.csv',   parse_dates=['datetime'])
df_test  = pd.read_csv('data/features_test.csv',  parse_dates=['datetime'])

# TTM model parameters
CONTEXT_LEN = 512    # 512-hour historical context window (~21 days)
HORIZON     = 24     # 24-hour forecast horizon
TARGET_COL  = 'demand_mw'

# TTMs work best with univariate signals, so we remove the noise (exogenous variables)

print(f'Target variable : {TARGET_COL}')
print(f'Train : {len(df_train):,}  Val : {len(df_val):,}  Test : {len(df_test):,}')

In [5]:
from sklearn.preprocessing import StandardScaler

# Standardize target channel
scaler = StandardScaler()
scaler.fit(df_train[[TARGET_COL]])

def scale_df(df):
    out = df.copy()
    out[[TARGET_COL]] = scaler.transform(df[[TARGET_COL]])
    return out

df_train_sc = scale_df(df_train)
df_val_sc   = scale_df(df_val)
df_test_sc  = scale_df(df_test)

# Concatenated history for zero-shot inference (train + val as context)
df_history = pd.concat([df_train_sc, df_val_sc], ignore_index=True)
print(f'History length for zero-shot: {len(df_history):,} rows')

In [6]:
TTM_MODEL_ID = 'ibm-granite/granite-timeseries-ttm-r2'

print(f'Loading {TTM_MODEL_ID}…')
zs_model = TinyTimeMixerForPrediction.from_pretrained(
    TTM_MODEL_ID,
    prediction_filter_length = HORIZON,
)
zs_model = zs_model.to(DEVICE)
zs_model.eval()

config = zs_model.config
print(f'Context length : {config.context_length}')
print(f'Forecast length: {config.prediction_length}')
print(f'Parameters     : {sum(p.numel() for p in zs_model.parameters()):,}')

## 3 · Build ForecastDFDatasets

In [7]:
def make_dataset(df, context_len=CONTEXT_LEN, horizon=HORIZON, stride=1):
    """Wrap a DataFrame in ForecastDFDataset expected by granite-tsfm."""
    return ForecastDFDataset(
        df,
        id_columns           = [],
        timestamp_column     = 'datetime',
        target_columns       = [TARGET_COL],
        conditional_columns  = [],
        context_length       = context_len,
        prediction_length    = horizon,
        stride               = stride,
    )

# Zero-shot test dataset: use last CONTEXT_LEN rows of history as prefix,
# then test data — ensures every prediction window falls within the test period.
df_for_zs_test = pd.concat([
    df_history.tail(CONTEXT_LEN).reset_index(drop=True),
    df_test_sc,
], ignore_index=True)

# stride=HORIZON gives non-overlapping daily predictions
test_dataset_zs = make_dataset(df_for_zs_test, stride=HORIZON)
print(f'Zero-shot test dataset: {len(test_dataset_zs)} windows')

# Few-shot fine-tuning dataset: sample 5% of windows uniformly across all 3.5 years (stride=24)
fs_train_ds = make_dataset(df_train_sc, stride=24)
fs_val_ds   = make_dataset(df_val_sc,   stride=HORIZON)
print(f'Few-shot train dataset  : {len(fs_train_ds)} windows (sampled across full 3.5-year history)')
print(f'Few-shot val dataset    : {len(fs_val_ds)} windows')

In [8]:
from torch.utils.data import DataLoader
from torch.utils.data.dataloader import default_collate

def collate_skip_timestamps(batch):
    """Drop pandas Timestamp fields that PyTorch's default collate cannot handle."""
    filtered = [
        {k: v for k, v in item.items() if not isinstance(v, pd.Timestamp)}
        for item in batch
    ]
    return default_collate(filtered)

t0  = time.time()
loader = DataLoader(test_dataset_zs, batch_size=64, shuffle=False,
                    collate_fn=collate_skip_timestamps)

preds_zs, trues_zs = [], []

with torch.no_grad():
    for batch in loader:
        past_values = batch['past_values'].to(DEVICE)
        outputs     = zs_model(past_values=past_values)
        preds_zs.append(outputs.prediction_outputs.cpu().numpy())
        trues_zs.append(batch['future_values'].numpy())

inference_time_zs = time.time() - t0

preds_zs = np.concatenate(preds_zs, axis=0)   # (n_windows, horizon, 1)
trues_zs = np.concatenate(trues_zs, axis=0)

# Extract target channel (index 0) and inverse-transform
target_mean = scaler.mean_[0]
target_std  = scaler.scale_[0]

y_pred_zs = preds_zs[:, :, 0].ravel() * target_std + target_mean
y_true_zs = trues_zs[:, :, 0].ravel() * target_std + target_mean

print(f'Inference time (zero-shot): {inference_time_zs:.2f}s')
print(f'Forecast points: {len(y_pred_zs):,}')

In [9]:
metrics_zs = du.compute_metrics(y_true_zs, y_pred_zs, label='TTM zero-shot')
metrics_zs['train_s']     = 0.0
metrics_zs['inference_s'] = inference_time_zs

## 5 · Few-shot fine-tuning (5% of training data)

In [10]:
# REMOVE/comment later
# debugging an issue with the TRainingArguments consutructor
import inspect
import transformers

print("Transformers version:", transformers.__version__)
print("TrainingArguments class :", TrainingArguments)
print("Defined in module       :", TrainingArguments.__module__)
try:
    print("Source file             :", inspect.getfile(TrainingArguments))
except Exception:
    pass
print("\nAccepted constructor parameters:")
print(inspect.signature(TrainingArguments.__init__))

In [14]:
# Load pre-trained model in native Channel-Independent mode (preserves zero-shot curve)
fs_model = TinyTimeMixerForPrediction.from_pretrained(
    TTM_MODEL_ID,
    prediction_filter_length = HORIZON,
)

# Gentle fine-tuning: allow small learning rate adaptation across the model
# (or train prediction head with pre-trained weights intact)
for param in fs_model.backbone.parameters():
    param.requires_grad = True

print(f'Trainable parameters: {sum(p.numel() for p in fs_model.parameters() if p.requires_grad):,}')

training_args = TrainingArguments(
    output_dir                  = 'saved_models/ttm_fewshot',
    num_train_epochs            = 10,
    per_device_train_batch_size = 64,
    per_device_eval_batch_size  = 64,
    learning_rate               = 2e-5,                 # Gentle learning rate for foundation adaptation
    lr_scheduler_type           = 'cosine',
    warmup_steps                = 10,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    save_total_limit            = 2,
    logging_steps               = 5,
    report_to                   = 'none',
    seed                        = 42,
)

trainer = Trainer(
    model        = fs_model,
    args         = training_args,
    train_dataset= fs_train_ds,
    eval_dataset = fs_val_ds,
    callbacks    = [
        EarlyStoppingCallback(
            early_stopping_patience  = 3,
            early_stopping_threshold = 0.0005,
        )
    ],
)

print(f'Fine-tuning on {len(fs_train_ds)} windows…')
t0 = time.time()
trainer.train()
finetune_time = time.time() - t0
print(f'Fine-tuning completed in {finetune_time:.1f}s  ({finetune_time/60:.1f} min)')

In [15]:
# Plot fine-tuning loss
log_history = trainer.state.log_history
train_losses = [e['loss'] for e in log_history if 'loss' in e and 'eval_loss' not in e]
eval_losses  = [e['eval_loss'] for e in log_history if 'eval_loss' in e]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label='Train loss')
if eval_losses:
    eval_x = np.linspace(0, len(train_losses)-1, len(eval_losses))
    ax.plot(eval_x, eval_losses, label='Val loss', linestyle='--')
ax.set_title('TTM few-shot fine-tuning loss')
ax.set_xlabel('Step')
ax.legend()
plt.tight_layout()
plt.savefig('data/fig_ttm_finetune.png', dpi=150, bbox_inches='tight')
plt.show()

In [16]:
fs_model.eval()
fs_model = fs_model.to(DEVICE)

# Re-use the same test dataset built for zero-shot
fs_loader = DataLoader(test_dataset_zs, batch_size=64, shuffle=False,
                       collate_fn=collate_skip_timestamps)


t0 = time.time()
preds_fs, trues_fs = [], []

with torch.no_grad():
    for batch in fs_loader:
        past_values = batch['past_values'].to(DEVICE)
        outputs     = fs_model(past_values=past_values)
        preds_fs.append(outputs.prediction_outputs.cpu().numpy())
        trues_fs.append(batch['future_values'].numpy())

inference_time_fs = time.time() - t0

preds_fs = np.concatenate(preds_fs, axis=0)
trues_fs = np.concatenate(trues_fs, axis=0)

y_pred_fs = preds_fs[:, :, 0].ravel() * target_std + target_mean
y_true_fs = trues_fs[:, :, 0].ravel() * target_std + target_mean

print(f'Inference time (few-shot): {inference_time_fs:.2f}s')
print(f'Forecast points: {len(y_pred_fs):,}')

In [24]:
metrics_fs = du.compute_metrics(y_true_fs, y_pred_fs, label='TTM few-shot')
metrics_fs['train_s']     = finetune_time
metrics_fs['inference_s'] = inference_time_fs

## 7 · Plots

In [23]:
# Build datetime array for test predictions
n_windows  = len(preds_zs)
dates_test = df_for_zs_test['datetime'].values

dates_zs = np.concatenate([
    dates_test[CONTEXT_LEN + i * HORIZON : CONTEXT_LEN + i * HORIZON + HORIZON]
    for i in range(n_windows)
])
dates_zs = pd.to_datetime(dates_zs)

In [19]:
du.plot_predictions(
    y_true_zs[:168], y_pred_zs[:168],
    title     = 'TTM zero-shot — first week of test set',
    dates     = dates_zs[:168],
    save_path = 'data/fig_ttm_zeroshot_week.png',
)

In [20]:
du.plot_predictions(
    y_true_fs[:168], y_pred_fs[:168],
    title     = 'TTM few-shot — first week of test set',
    dates     = dates_zs[:168],
    save_path = 'data/fig_ttm_fewshot_week.png',
)

In [21]:
# Side-by-side zero-shot vs few-shot (one representative week)
idx = slice(0, 168)
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, y_pred, label, color in [
    (axes[0], y_pred_zs[idx], 'Zero-shot', '#ff7f0e'),
    (axes[1], y_pred_fs[idx], 'Few-shot',  '#2ca02c'),
]:
    ax.plot(dates_zs[idx], y_true_zs[idx], label='Actual',  color='#1f77b4', linewidth=1.2)
    ax.plot(dates_zs[idx], y_pred,          label=label,   color=color, linewidth=1.2, linestyle='--')
    ax.set_title(f'TTM {label}')
    ax.set_ylabel('Demand (MW)')
    ax.legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.tight_layout()
plt.savefig('data/fig_ttm_comparison_week.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 · Save predictions and metrics

In [22]:
# Zero-shot
pd.DataFrame({'datetime': dates_zs, 'y_true': y_true_zs, 'y_pred': y_pred_zs}).to_csv(
    'data/predictions_ttm_zeroshot.csv', index=False
)
with open('data/metrics_ttm_zeroshot.json', 'w') as f:
    json.dump(metrics_zs, f, indent=2)

# Few-shot
pd.DataFrame({'datetime': dates_zs, 'y_true': y_true_fs, 'y_pred': y_pred_fs}).to_csv(
    'data/predictions_ttm_fewshot.csv', index=False
)
with open('data/metrics_ttm_fewshot.json', 'w') as f:
    json.dump(metrics_fs, f, indent=2)

print('Saved predictions and metrics for both TTM modes.')
print('\nZero-shot metrics:', metrics_zs)
print('Few-shot  metrics:', metrics_fs)

# Sync outputs to Google Drive (if in Colab)
du.sync_to_drive()


## Summary

| Mode | MAPE (%) | RMSE (MW) | MAE (MW) | Fine-tune time |
|------|----------|-----------|----------|----------------|
| Zero-shot | … | … | … | 0 s |
| Few-shot  | … | … | … | … s |

Next step → `fase3_validacion_y_comparativa.ipynb`

In [25]:
#Terminate runtime
from google.colab import runtime
runtime.unassign()